# Few-Shot Ordinal-Aware CLIP for Diabetic Retinopathy Grading
## MMRDR Benchmark — Full Training & Evaluation Notebook

This notebook provides a **complete, reproducible pipeline** for the few-shot ordinal DR grading paper:

| Step | Description |
|------|-------------|
| **1** | Environment setup & imports |
| **2** | Experiment grid definition |
| **3** | Dataset overview & class distribution plots |
| **4** | Prepare few-shot splits |
| **5** | Train all experiments (modality × k-shot × seed) |
| **6** | Parse & aggregate results |
| **7** | Paper-quality plots: k-shot curves, confusion matrices, ablation table |

---
**Modalities**: CFP · OCT · UWF  
**k-shots**: 0 (zero-shot), 1, 2, 4, 8, 16, 32  
**Seeds**: 1 · 2 · 3 · 4 · 5  
**Metrics**: QWK · MAE · Macro-F1 · AUC · Accuracy · LOS

## 1 · Environment Setup

In [ ]:
import os, sys, json, warnings, logging
import subprocess
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from omegaconf import OmegaConf
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

# ── Locate project root (contains 'configs/' dir) ────────────────────────────
ROOT = Path.cwd()
while not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from scripts.train import load_config
from src.data.datamodule import FewShotDataModule
from src.models.runner import FewShotOrdinalRunner

RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root : {ROOT}')
print(f'Device       : {DEVICE}')
print(f'Results dir  : {RESULTS}')
print(f'PyTorch      : {torch.__version__}')
print(f'PL           : {pl.__version__}')

## 2 · Experiment Grid

Edit these cells to control what gets trained / re-run.

In [ ]:
# ── Modality → config file + number of DR grades ─────────────────────────────
MODALITIES = {
    'CFP': {'config': 'configs/mrdr_cfp.yaml', 'num_ranks': 5},
    'OCT': {'config': 'configs/mrdr_oct.yaml', 'num_ranks': 3},
    'UWF': {'config': 'configs/mrdr_uwf.yaml', 'num_ranks': 5},
}

# ── k-shot regimes (0 = zero-shot, no training) ───────────────────────────────
K_SHOTS = [0, 1, 2, 4, 8, 16, 32]

# ── Seeds for statistical reliability ────────────────────────────────────────
SEEDS = [1, 2, 3, 4, 5]

# ── Per-run epoch budget (100 is appropriate for few-shot) ───────────────────
EPOCHS = 100

# ── Ablation variants (empty dict = full model) ───────────────────────────────
ABLATIONS = {
    'Ours (Full)':         {},
    'w/o RankFormer':      {'model.prompt_learner.name': 'ordinalclip_rank_prompt_learner'},
    'w/o SMS':             {'model.sms.enabled': 'false'},
    'w/o PairwiseLoss':    {'loss.mu_pairwise': '0.0'},
    'w/o RankingLoss':     {'loss.lambda_rank': '0.0'},
    'w/o LoRA':            {'model.image_encoder.lora.enabled': 'false'},
}

# ── Quick count ───────────────────────────────────────────────────────────────
fs_k   = [k for k in K_SHOTS if k > 0]
zs_runs = len(MODALITIES)
fs_runs = len(MODALITIES) * len(fs_k) * len(SEEDS)
print(f'Modalities  : {list(MODALITIES)}')
print(f'k-shots     : {K_SHOTS}  (0 = zero-shot)')
print(f'Seeds       : {SEEDS}')
print(f'Zero-shot   : {zs_runs} run(s)')
print(f'Few-shot    : {fs_runs} runs  (× {len(ABLATIONS)} variants = {fs_runs * len(ABLATIONS)} for full ablation)')

## 3 · Dataset Overview

Load manifests and plot class distributions per modality.

In [ ]:
def load_manifest(modality: str) -> pd.DataFrame | None:
    path = ROOT / 'data' / 'MMRDR' / f'{modality}_manifest.csv'
    if not path.exists():
        print(f'  [SKIP] manifest not found: {path}')
        return None
    return pd.read_csv(path)

manifests = {m: load_manifest(m) for m in MODALITIES}

# ── Plot class distributions ──────────────────────────────────────────────────
PALETTE = ['#4361EE', '#3A0CA3', '#7209B7', '#F72585', '#FF9E00',
           '#06D6A0', '#118AB2', '#EF233C']
GRADE_NAMES = ['Grade 0\n(No DR)', 'Grade 1\n(Mild)', 'Grade 2\n(Moderate)',
               'Grade 3\n(Severe)', 'Grade 4\n(Proliferative)']

available = {m: df for m, df in manifests.items() if df is not None}

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4.5),
                             constrained_layout=True)
    if len(available) == 1:
        axes = [axes]

    for ax, (mod, df) in zip(axes, available.items()):
        num_ranks = MODALITIES[mod]['num_ranks']
        for split, style in [('train', 0.8), ('test', 0.45)]:
            sub = df[df['split'] == split]
            counts = sub['grade'].value_counts().sort_index()
            bars = ax.bar(counts.index, counts.values,
                          color=[PALETTE[i] for i in counts.index],
                          alpha=style, edgecolor='white', linewidth=0.8,
                          label=split.capitalize())
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bar.get_width() / 2, h + 8, f'{int(h):,}',
                        ha='center', va='bottom', fontsize=7, color='#333')

        ax.set_title(f'MMRDR-{mod}', fontsize=13, fontweight='bold', pad=10)
        ax.set_xlabel('DR Grade', fontsize=10)
        ax.set_ylabel('Image Count', fontsize=10)
        ax.set_xticks(range(num_ranks))
        ax.set_xticklabels(GRADE_NAMES[:num_ranks], fontsize=8)
        ax.legend(fontsize=9, framealpha=0.7)
        ax.spines[['top', 'right']].set_visible(False)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    fig.suptitle('MMRDR Dataset — Class Distributions by Modality & Split',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.savefig(FIGURES / 'dataset_distribution.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved → {FIGURES / "dataset_distribution.pdf"}')
else:
    print('No manifests found. Run scripts/prepare_mmrdr.py first.')

## 4 · Prepare Few-Shot Splits

Generates support/query text files for each modality × k × seed combination.

In [ ]:
SPLITS_DIR = ROOT / 'data' / 'MMRDR' / 'splits'

def splits_exist(modality: str, k: int, seed: int) -> bool:
    mod_lower = modality.lower()
    sup = SPLITS_DIR / f'{mod_lower}_{k}shot_s{seed}_support.txt'
    qry = SPLITS_DIR / f'{mod_lower}_{k}shot_s{seed}_query.txt'
    return sup.exists() and qry.exists()

to_generate = [
    m for m in MODALITIES
    if manifests.get(m) is not None
    and not all(splits_exist(m, k, s) for k in fs_k for s in SEEDS)
]

if to_generate:
    print('Generating splits for:', to_generate)
    kshots_str = ','.join(str(k) for k in fs_k)
    seeds_str  = ','.join(str(s) for s in SEEDS)
    for mod in to_generate:
        cmd = [
            sys.executable, 'src/data/build_splits.py',
            '--modality', mod,
            '--kshots', kshots_str,
            '--seeds', seeds_str,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print('ERROR:', result.stderr)
else:
    print('All splits already exist — skipping generation.')

## 5 · Training

### 5a · Single-run helper

This function trains (or skips if already done) one experiment and returns the test metrics.

In [ ]:
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

def run_experiment(
    config_path: str,
    modality: str,
    k_shot: int,
    seed: int,
    variant_name: str = 'Ours (Full)',
    extra_overrides: dict | None = None,
    force_retrain: bool = False,
) -> dict:
    """
    Train and evaluate one experiment.

    Returns a dict of metrics, or loads cached results if they already exist.
    """
    variant_slug = variant_name.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
    out_dir = RESULTS / variant_slug / f'{modality}_{k_shot}shot_s{seed}'
    stats_file = out_dir / 'test_stats.json'

    # ── Return cached results ─────────────────────────────────────────────
    if stats_file.exists() and not force_retrain:
        with open(stats_file) as f:
            return json.load(f)

    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Build OmegaConf ───────────────────────────────────────────────────
    overrides = [f'data.k_shot={k_shot}', f'data.seed={seed}',
                 f'train.epochs={EPOCHS}']
    if extra_overrides:
        overrides += [f'{k}={v}' for k, v in extra_overrides.items()]

    cfg = load_config(config_path, overrides)

    pl.seed_everything(seed, workers=True)

    datamodule = FewShotDataModule(cfg)
    model = FewShotOrdinalRunner(cfg)

    # ── Callbacks ─────────────────────────────────────────────────────────
    monitor  = cfg.train.get('early_stopping_monitor', 'val_qwk')
    mode     = cfg.train.get('early_stopping_mode', 'max')
    patience = cfg.train.get('early_stopping_patience', 20)

    callbacks = [
        EarlyStopping(monitor=monitor, patience=patience, mode=mode, verbose=False),
        ModelCheckpoint(dirpath=str(out_dir), filename='best',
                        monitor=monitor, mode=mode, save_top_k=1),
    ]

    # ── Trainer ───────────────────────────────────────────────────────────
    trainer = pl.Trainer(
        max_epochs=EPOCHS if k_shot > 0 else 0,
        accelerator='auto', devices=1,
        precision='16-mixed',
        callbacks=callbacks,
        log_every_n_steps=1,
        enable_checkpointing=True,
        enable_progress_bar=True,
        logger=False,
        enable_model_summary=False,
    )

    if k_shot > 0:
        trainer.fit(model, datamodule=datamodule)
        # Load best checkpoint before testing
        best_ckpt = callbacks[1].best_model_path
        if best_ckpt:
            model = FewShotOrdinalRunner.load_from_checkpoint(best_ckpt, cfg=cfg)

    trainer.test(model, datamodule=datamodule, verbose=False)

    # ── Cache & return ────────────────────────────────────────────────────
    metrics = {k: float(v) for k, v in model.latest_metrics.items()}
    with open(stats_file, 'w') as f:
        json.dump(metrics, f, indent=4)

    return metrics

print('run_experiment() helper defined.')

### 5b · Run Full Experiment Grid (Main Model)

> **Tip**: Set `MODALITIES_TO_RUN` and `K_SHOTS_TO_RUN` to subset the grid for a quick test.

In [ ]:
# ── Subset these to run only part of the grid ─────────────────────────────────
MODALITIES_TO_RUN = list(MODALITIES)      # e.g. ['CFP'] for a single modality
K_SHOTS_TO_RUN    = K_SHOTS               # e.g. [0, 4] for quick test
SEEDS_TO_RUN      = SEEDS

# ── Collect all results ───────────────────────────────────────────────────────
all_results = []  # list of dicts

total = len(MODALITIES_TO_RUN) * len(K_SHOTS_TO_RUN) * (
    1 if 0 in K_SHOTS_TO_RUN else 0  # zero-shot = 1 run per modality
) + len(MODALITIES_TO_RUN) * len([k for k in K_SHOTS_TO_RUN if k > 0]) * len(SEEDS_TO_RUN)

pbar = tqdm(total=total, desc='Experiments')

seen_zeroshot = set()  # zero-shot only once per modality

for mod in MODALITIES_TO_RUN:
    cfg_path = MODALITIES[mod]['config']

    for k in K_SHOTS_TO_RUN:
        seeds_for_k = [1] if k == 0 else SEEDS_TO_RUN  # zero-shot: single deterministic run

        for seed in seeds_for_k:
            if k == 0 and mod in seen_zeroshot:
                continue
            if k == 0:
                seen_zeroshot.add(mod)

            pbar.set_postfix({'mod': mod, 'k': k, 'seed': seed})

            metrics = run_experiment(
                config_path=cfg_path,
                modality=mod,
                k_shot=k,
                seed=seed,
                variant_name='Ours (Full)',
            )

            all_results.append({
                'Variant':  'Ours (Full)',
                'Modality': mod,
                'k':        k,
                'Seed':     seed,
                **metrics,
            })
            pbar.update(1)

pbar.close()

df_results = pd.DataFrame(all_results)
print('\nDone! Shape:', df_results.shape)
df_results.head(10)

### 5c · Run Ablation Study

Runs CFP / k=4 / all seeds for each ablation variant. Skip if not needed.

In [ ]:
RUN_ABLATIONS   = True    # set False to skip
ABLATION_MOD    = 'CFP'
ABLATION_K      = 4
ABLATION_SEEDS  = SEEDS

ablation_results = []

if RUN_ABLATIONS:
    cfg_path = MODALITIES[ABLATION_MOD]['config']
    total_abl = len(ABLATIONS) * len(ABLATION_SEEDS)
    pbar = tqdm(total=total_abl, desc='Ablations')

    for variant_name, overrides in ABLATIONS.items():
        for seed in ABLATION_SEEDS:
            pbar.set_postfix({'variant': variant_name, 'seed': seed})
            metrics = run_experiment(
                config_path=cfg_path,
                modality=ABLATION_MOD,
                k_shot=ABLATION_K,
                seed=seed,
                variant_name=variant_name,
                extra_overrides=overrides,
            )
            ablation_results.append({
                'Variant':  variant_name,
                'Modality': ABLATION_MOD,
                'k':        ABLATION_K,
                'Seed':     seed,
                **metrics,
            })
            pbar.update(1)
    pbar.close()

df_ablation = pd.DataFrame(ablation_results)
print('Ablation shape:', df_ablation.shape)
df_ablation.head()

## 6 · Aggregate & Display Results Table

In [ ]:
METRICS = ['qwk', 'mae', 'f1', 'auc', 'acc', 'los']
METRIC_LABELS = {'qwk': 'QWK↑', 'mae': 'MAE↓', 'f1': 'Macro-F1↑',
                 'auc': 'AUC↑', 'acc': 'Accuracy↑', 'los': 'LOS↑'}

def aggregate(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Compute mean ± std per group."""
    rows = []
    for keys, grp in df.groupby(group_cols):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else [keys]))
        for m in METRICS:
            if m in grp.columns:
                row[f'{m}_mean'] = grp[m].mean()
                row[f'{m}_std']  = grp[m].std(ddof=1) if len(grp) > 1 else 0.0
                row[m] = f"{grp[m].mean():.4f} ± {row[f'{m}_std']:.4f}"
        rows.append(row)
    return pd.DataFrame(rows)

# ── Main results table ────────────────────────────────────────────────────────
if not df_results.empty:
    df_agg = aggregate(df_results, ['Variant', 'Modality', 'k'])
    display_cols = ['Variant', 'Modality', 'k'] + METRICS
    display(df_agg[display_cols].sort_values(['Modality', 'k']).to_string(index=False))

    # Save
    df_agg.to_csv(RESULTS / 'main_results.csv', index=False)
    print(f'\nSaved → {RESULTS / "main_results.csv"}')

## 7 · Paper-Quality Plots

### 7a · k-Shot Performance Curves

In [ ]:
# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

MOD_COLORS  = {'CFP': '#4361EE', 'OCT': '#F72585', 'UWF': '#06D6A0'}
MOD_MARKERS = {'CFP': 'o',       'OCT': 's',       'UWF': '^'}

PLOT_METRICS = ['qwk', 'mae', 'f1', 'auc']
PLOT_LABELS  = ['QWK ↑', 'MAE ↓', 'Macro-F1 ↑', 'AUC ↑']

if not df_results.empty:
    df_agg = aggregate(df_results, ['Variant', 'Modality', 'k'])
    main = df_agg[df_agg['Variant'] == 'Ours (Full)']

    fig, axes = plt.subplots(1, len(PLOT_METRICS), figsize=(5 * len(PLOT_METRICS), 4.2),
                             constrained_layout=True)

    for ax, metric, label in zip(axes, PLOT_METRICS, PLOT_LABELS):
        for mod in MODALITIES_TO_RUN:
            sub = main[main['Modality'] == mod].sort_values('k')
            if sub.empty or f'{metric}_mean' not in sub.columns:
                continue
            mu  = sub[f'{metric}_mean'].values
            std = sub[f'{metric}_std'].values
            ks  = sub['k'].values

            ax.plot(ks, mu, marker=MOD_MARKERS[mod], color=MOD_COLORS[mod],
                    linewidth=2, markersize=6, label=f'MMRDR-{mod}')
            ax.fill_between(ks, mu - std, mu + std,
                            alpha=0.15, color=MOD_COLORS[mod])

        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.set_xlabel('k (support shots per grade)', fontsize=9)
        ax.set_xticks(K_SHOTS_TO_RUN)
        ax.set_xticklabels([str(k) if k > 0 else '0\n(ZS)' for k in K_SHOTS_TO_RUN], fontsize=8)
        if metric in ('qwk', 'f1', 'auc', 'acc', 'los'):
            ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8, framealpha=0.8)

    fig.suptitle('Few-Shot DR Grading Performance vs. k-Shot Regime',
                 fontsize=13, fontweight='bold')
    plt.savefig(FIGURES / 'kshot_curves.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved → {FIGURES / "kshot_curves.pdf"}')
else:
    print('No results yet — run Section 5 first.')

### 7b · Per-Modality Bar Chart (k = 4 and k = 16)

In [ ]:
BAR_K     = [4, 16]
BAR_METS  = ['qwk', 'f1', 'auc', 'acc']
BAR_LABS  = ['QWK', 'Macro-F1', 'AUC', 'Accuracy']

if not df_results.empty:
    df_agg = aggregate(df_results, ['Variant', 'Modality', 'k'])
    main = df_agg[(df_agg['Variant'] == 'Ours (Full)') & (df_agg['k'].isin(BAR_K))]

    n_mets = len(BAR_METS)
    n_k    = len(BAR_K)
    mods   = [m for m in MODALITIES_TO_RUN]
    x      = np.arange(len(mods))
    width  = 0.35 / n_k

    k_colors = {4: '#4361EE', 16: '#F72585', 32: '#06D6A0'}

    fig, axes = plt.subplots(1, n_mets, figsize=(4.5 * n_mets, 4.2),
                             constrained_layout=True)

    for ax, metric, label in zip(axes, BAR_METS, BAR_LABS):
        for i, k in enumerate(BAR_K):
            sub = main[main['k'] == k].set_index('Modality')
            vals = [sub.loc[m, f'{metric}_mean'] if m in sub.index else 0 for m in mods]
            errs = [sub.loc[m, f'{metric}_std']  if m in sub.index else 0 for m in mods]
            offset = (i - (n_k - 1) / 2) * width * 2.2
            bars = ax.bar(x + offset, vals, width * 1.8,
                          yerr=errs, capsize=3,
                          color=k_colors.get(k, '#888'),
                          alpha=0.85, label=f'k={k}',
                          edgecolor='white', linewidth=0.5,
                          error_kw={'elinewidth': 1.2, 'ecolor': '#333'})
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + max(errs) + 0.005,
                        f'{v:.3f}', ha='center', va='bottom',
                        fontsize=7, color='#222')

        ax.set_title(label, fontsize=11, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([f'MMRDR-{m}' for m in mods], fontsize=9)
        ax.set_ylim(0, 1.12)
        ax.legend(fontsize=8, framealpha=0.8)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

    fig.suptitle('Per-Modality Performance at Selected k-Shot Regimes',
                 fontsize=13, fontweight='bold')
    plt.savefig(FIGURES / 'modality_bar.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved → {FIGURES / "modality_bar.pdf"}')
else:
    print('No results yet.')

### 7c · Confusion Matrix (best seed, chosen modality & k)

In [ ]:
CM_MOD   = 'CFP'
CM_K     = 16
CM_SEED  = 1

def get_predictions(config_path, modality, k_shot, seed, variant_name='Ours (Full)',
                    extra_overrides=None):
    """Re-run test pass and collect raw predictions for a confusion matrix."""
    variant_slug = variant_name.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
    out_dir = RESULTS / variant_slug / f'{modality}_{k_shot}shot_s{seed}'
    ckpt    = out_dir / 'best.ckpt'

    overrides = [f'data.k_shot={k_shot}', f'data.seed={seed}',
                 f'train.epochs={EPOCHS}']
    if extra_overrides:
        overrides += [f'{k}={v}' for k, v in extra_overrides.items()]

    cfg = load_config(config_path, overrides)
    datamodule = FewShotDataModule(cfg)

    if ckpt.exists():
        model = FewShotOrdinalRunner.load_from_checkpoint(str(ckpt), cfg=cfg)
    else:
        model = FewShotOrdinalRunner(cfg)

    trainer = pl.Trainer(accelerator='auto', devices=1, precision='16-mixed',
                         logger=False, enable_progress_bar=True,
                         enable_model_summary=False)
    trainer.test(model, datamodule=datamodule, verbose=False)

    # val_preds/targets were cleared after test — collect from test step manually
    # They are in model.latest_metrics (scalars). For the CM we need raw outputs.
    # Re-collect by running predict loop:
    import torch.nn.functional as F
    datamodule.setup('test')
    loader = datamodule.test_dataloader()
    all_preds, all_targets = [], []
    model.eval().to(DEVICE)
    with torch.no_grad():
        for batch in loader:
            imgs, labels, _, _ = batch
            imgs = imgs.to(DEVICE)
            img_f, txt_f = model(imgs)
            scale = model.clip_model.logit_scale.exp()
            sim   = scale * img_f @ txt_f.T
            if model.sms is not None:
                sim = model.sms(sim)
            preds = F.softmax(sim, dim=-1).argmax(dim=-1).cpu()
            all_preds.append(preds)
            all_targets.append(labels)
    return torch.cat(all_targets).numpy(), torch.cat(all_preds).numpy()

if CM_MOD in MODALITIES_TO_RUN and CM_K in K_SHOTS_TO_RUN:
    cfg_path   = MODALITIES[CM_MOD]['config']
    num_ranks  = MODALITIES[CM_MOD]['num_ranks']

    targets, preds = get_predictions(cfg_path, CM_MOD, CM_K, CM_SEED)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

    # Raw count CM
    cm_counts = confusion_matrix(targets, preds)
    disp = ConfusionMatrixDisplay(cm_counts,
                                  display_labels=[f'G{i}' for i in range(num_ranks)])
    disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'Confusion Matrix (counts)\nMMRDR-{CM_MOD}  k={CM_K}  seed={CM_SEED}',
                      fontsize=11, fontweight='bold')

    # Row-normalised CM
    cm_norm = cm_counts.astype(float) / cm_counts.sum(axis=1, keepdims=True)
    disp2 = ConfusionMatrixDisplay(np.round(cm_norm, 2),
                                   display_labels=[f'G{i}' for i in range(num_ranks)])
    disp2.plot(ax=axes[1], cmap='Purples', colorbar=False)
    axes[1].set_title(f'Confusion Matrix (row-normalised)\nMMRDR-{CM_MOD}  k={CM_K}  seed={CM_SEED}',
                      fontsize=11, fontweight='bold')

    plt.savefig(FIGURES / f'confusion_{CM_MOD}_k{CM_K}_s{CM_SEED}.pdf',
                bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved → {FIGURES}')
else:
    print(f'Skipping CM: {CM_MOD} k={CM_K} not in run grid.')

### 7d · Ablation Study Table & Plot

In [ ]:
if not df_ablation.empty:
    df_abl_agg = aggregate(df_ablation, ['Variant', 'Modality', 'k'])

    # ── Nicely formatted table ────────────────────────────────────────────
    display_cols = ['Variant'] + METRICS
    print('Ablation Study — MMRDR-CFP, k=4 shots (mean ± std over 5 seeds)')
    print('=' * 80)
    display(df_abl_agg[display_cols])
    df_abl_agg[display_cols].to_csv(RESULTS / 'ablation_results.csv', index=False)

    # ── Horizontal bar chart: QWK ─────────────────────────────────────────
    abl_mods = list(ABLATIONS.keys())
    full_idx  = abl_mods.index('Ours (Full)')
    qwk_vals  = []
    qwk_errs  = []
    for v in abl_mods:
        row = df_abl_agg[df_abl_agg['Variant'] == v]
        if row.empty or 'qwk_mean' not in row.columns:
            qwk_vals.append(0); qwk_errs.append(0)
        else:
            qwk_vals.append(row['qwk_mean'].values[0])
            qwk_errs.append(row['qwk_std'].values[0])

    colors = ['#4361EE' if i == full_idx else '#ADB5BD' for i in range(len(abl_mods))]
    y_pos  = np.arange(len(abl_mods))

    fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
    bars = ax.barh(y_pos, qwk_vals, xerr=qwk_errs, height=0.5,
                   color=colors, edgecolor='white', capsize=3,
                   error_kw={'elinewidth': 1.4, 'ecolor': '#333'})
    for bar, v in zip(bars, qwk_vals):
        ax.text(v + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{v:.4f}', va='center', fontsize=9)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(abl_mods, fontsize=10)
    ax.set_xlabel('QWK ↑', fontsize=11)
    ax.set_title(f'Ablation Study — MMRDR-{ABLATION_MOD}  k={ABLATION_K}',
                 fontsize=12, fontweight='bold')
    ax.set_xlim(0, max(qwk_vals) * 1.2 + 0.05)
    ax.axvline(qwk_vals[full_idx], color='#4361EE', linestyle='--',
               linewidth=1.2, alpha=0.6)

    plt.savefig(FIGURES / 'ablation_qwk.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Saved → {FIGURES / "ablation_qwk.pdf"}')
else:
    print('No ablation results yet — run Section 5c first.')

### 7e · Radar / Spider Chart — Per-Metric Overview at k=16

In [ ]:
RADAR_K = 16
RADAR_METRICS = ['qwk', 'f1', 'auc', 'acc', 'los']
RADAR_LABELS  = ['QWK', 'F1', 'AUC', 'Acc', 'LOS']

if not df_results.empty:
    df_agg = aggregate(df_results, ['Variant', 'Modality', 'k'])
    radar_data = df_agg[(df_agg['Variant'] == 'Ours (Full)') & (df_agg['k'] == RADAR_K)]

    if radar_data.empty:
        print(f'No data at k={RADAR_K}; adjust RADAR_K.')
    else:
        N = len(RADAR_METRICS)
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
        angles += angles[:1]  # close polygon

        fig, ax = plt.subplots(figsize=(5, 5),
                               subplot_kw={'projection': 'polar'},
                               constrained_layout=True)

        for mod in MODALITIES_TO_RUN:
            row = radar_data[radar_data['Modality'] == mod]
            if row.empty:
                continue
            vals = [row[f'{m}_mean'].values[0] for m in RADAR_METRICS]
            vals += vals[:1]
            ax.plot(angles, vals, color=MOD_COLORS[mod], linewidth=2,
                    marker=MOD_MARKERS[mod], markersize=6, label=f'MMRDR-{mod}')
            ax.fill(angles, vals, color=MOD_COLORS[mod], alpha=0.1)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(RADAR_LABELS, fontsize=11, fontweight='bold')
        ax.set_ylim(0, 1)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=7, color='#888')
        ax.set_title(f'Multi-Metric Radar — k={RADAR_K} shots',
                     fontsize=12, fontweight='bold', pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

        plt.savefig(FIGURES / f'radar_k{RADAR_K}.pdf', bbox_inches='tight', dpi=300)
        plt.show()
        print(f'Saved → {FIGURES / f"radar_k{RADAR_K}.pdf"}')
else:
    print('No results yet.')

### 7f · Training Dynamics (Loss & Validation QWK vs. Epoch)

For this plot, enable Lightning CSV logging during training by setting `logger=True` in `run_experiment()` and then reading the `metrics.csv` file.

In [ ]:
# Reads Lightning's default CSV logger output if available
DYNAMICS_MOD  = 'CFP'
DYNAMICS_K    = 4
DYNAMICS_SEED = 1
VARIANT_SLUG  = 'Ours_Full_'

lightning_logs = ROOT / 'lightning_logs'

if lightning_logs.exists():
    versions = sorted(lightning_logs.glob('version_*/metrics.csv'))
    if versions:
        # Use the most recent version (last run)
        df_log = pd.read_csv(versions[-1])
        print(f'Loaded: {versions[-1]}')
        print(df_log.columns.tolist())

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)

        for col, ax, label in [
            ('train_loss', axes[0], 'Training Loss'),
            ('val_qwk',   axes[1], 'Validation QWK'),
        ]:
            if col in df_log.columns:
                sub = df_log[['epoch', col]].dropna()
                ax.plot(sub['epoch'], sub[col], color='#4361EE', linewidth=2)
                ax.set_xlabel('Epoch', fontsize=10)
                ax.set_title(label, fontsize=11, fontweight='bold')
                ax.spines[['top', 'right']].set_visible(False)

        fig.suptitle(f'Training Dynamics — MMRDR-{DYNAMICS_MOD}  k={DYNAMICS_K}  seed={DYNAMICS_SEED}',
                     fontsize=12, fontweight='bold')
        plt.savefig(FIGURES / 'training_dynamics.pdf', bbox_inches='tight', dpi=300)
        plt.show()
    else:
        print('No metrics.csv found in lightning_logs. Enable CSV logger to generate.')
else:
    print('lightning_logs/ not found. Skip this plot or enable CSV logging in run_experiment().')

## 8 · Final Summary Table (LaTeX-ready)

Generates a LaTeX table for direct copy-paste into the paper.

In [ ]:
if not df_results.empty:
    df_agg = aggregate(df_results, ['Variant', 'Modality', 'k'])
    main = df_agg[df_agg['Variant'] == 'Ours (Full)'].sort_values(['Modality', 'k'])

    # ── Build LaTeX rows ──────────────────────────────────────────────────
    print('% ─── LaTeX Table ───────────────────────────────────────────────')
    print(r'\begin{table}[ht]')
    print(r'\centering')
    print(r'\caption{Few-Shot DR Grading on MMRDR (mean $\pm$ std over 5 seeds)}')
    print(r'\label{tab:main}')
    print(r'\begin{tabular}{llrcccc}')
    print(r'\toprule')
    print(r'Modality & k & QWK $\uparrow$ & MAE $\downarrow$ & F1 $\uparrow$ & AUC $\uparrow$ \\')
    print(r'\midrule')

    prev_mod = None
    for _, row in main.iterrows():
        mod_str = f'MMRDR-{row["Modality"]}' if row['Modality'] != prev_mod else ''
        prev_mod = row['Modality']
        k_str = 'ZS' if row['k'] == 0 else str(int(row['k']))
        qwk  = f"{row.get('qwk_mean', 0):.3f}$\\pm${row.get('qwk_std', 0):.3f}"
        mae  = f"{row.get('mae_mean', 0):.3f}$\\pm${row.get('mae_std', 0):.3f}"
        f1   = f"{row.get('f1_mean',  0):.3f}$\\pm${row.get('f1_std',  0):.3f}"
        auc  = f"{row.get('auc_mean', 0):.3f}$\\pm${row.get('auc_std', 0):.3f}"
        print(f'{mod_str} & {k_str} & {qwk} & {mae} & {f1} & {auc} \\\\')
    print(r'\bottomrule')
    print(r'\end{tabular}')
    print(r'\end{table}')
    print('% ────────────────────────────────────────────────────────────────')
else:
    print('No results yet — run Section 5b first.')

## 9 · Quick Sanity Check (single forward pass)

Useful to confirm the model loads and runs before committing to a full sweep.

In [ ]:
import torch.nn.functional as F

SANITY_MOD = 'CFP'
SANITY_K   = 4

cfg_path = MODALITIES[SANITY_MOD]['config']
cfg = load_config(cfg_path, [f'data.k_shot={SANITY_K}', 'data.seed=1'])

model = FewShotOrdinalRunner(cfg).to(DEVICE)
model.eval()

dummy = torch.randn(4, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    img_f, txt_f = model(dummy)
    scale = model.clip_model.logit_scale.exp()
    sim   = scale * img_f @ txt_f.T
    probs = F.softmax(sim, dim=-1)
    preds = probs.argmax(dim=-1)

print(f'Image features shape : {img_f.shape}')
print(f'Text  features shape : {txt_f.shape}')
print(f'Similarity matrix    : {sim.shape}')
print(f'Probabilities        : {probs.round(decimals=4)}')
print(f'Predictions          : {preds.tolist()}')
print('\n✅ Sanity check passed!')